# Feature Engineering: Encoding Substructures
Scenario: You are investigating toxicity and its dependency on molecular structure. Using the LD50 Toxicity dataset by Zhu (as provided here: https://huggingface.co/datasets/scikit-fingerprints/TDC_ld50_zhu/viewer), you calculate various molecular descriptors based on the SMILES and train a Random Forest Regressor to predict toxicity. You want to see if there is an improvement of the model when you take into account chemical structure - and generally, whether chemical structure is generally related to toxicity.

Note: the dataset is based on sparse chemical space and will not deliver super high prediction scores.

#### Tasks:
1) Load and inspect the dataset `tdc_ld50_zhu.csv`.
2) Feature engineering: Create a feature matrix `X_md` by calculating descriptors (full list as in rdkit) (Snippet provided)

3) Finding common substructures (functional groups, common scaffolds) is an alternative to detecting similarities based on fingerprints. Use the provided snippets to create different structure-based features and use one-hot encoding to make them more accessible also for other models.

4) Compare the prediction performance of the model for each of the feature matrices and combinations thereof:
- X_md: molecular descriptors only
- X_scaf: One-Hot-encoded Murcko scaffolds
- X_fg: One-hot-encoded functional groups
- X_md_scaf: Concatenation of X_md and X_scaf
- X_md_scaf_fg: Concatenation of X_md, X_scaf and X_fg

Note: For every different dataset you have to perform a train-test split and train the model before predicting. You do not need to use the suggested names!

5) Pick the best one and remove low-variance features (threshold 1%), as well as highly correlated ones (threshold 90%). See if the performance of the predictions by the model improves. Hint: Use `VarianceThreshold` (Unsupervised Algorithm from Scikit-learn) and `corr_matrix = X.corr().abs()` to do so (e.g. similarly as used in the Clustering example on the ESOL dataset)

6) Calculate Morgan Fingerprints (radius 2, 2048 bit) from the SMILES strings via mol objects. Make sure not to use the outdated version. You can use either the dataframe, numpy arrays or simple lists for the fingerprints.

7) Use Butina clustering on UMAP visualisation of the smiles-fingerprint space (scatterplot) and compare the plot side by side (e.g. as suplots) with a scatterplot colourmapped to the toxicity (look at options such as `colorbar` for matplotlib for better visualisation). 

8) Repeat 6 and 7 for the fingerprints based on the Murcko Scaffolds from task 3

9) Respond to the discussion points


Import dependencies and datasets

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MolFromSmarts
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder

from sklearn.manifold import TSNE
import umap

import matplotlib.pyplot as plt
import seaborn as sns

from rdkit.ML.Cluster import Butina

1) Load and investigate the data

In [2]:
tox = pd.read_csv("tdc_ld50_zhu.csv")
tox.head()

,smiles,ld_50
0,[O-][N+](=Nc1ccccc1)c1ccccc1,2.505
1,BrC(Br)Br,2.343
2,C=CBr,2.330
3,Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br,1.465
4,S=C=Nc1ccc(Br)cc1,2.729


2) Calculate descriptors

In [4]:
descriptor_names = [d[0] for d in Descriptors._descList]

def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return [desc[1](mol) for desc in Descriptors._descList]

X_md = tox.smiles.apply(calc_descriptors)

X_md = pd.DataFrame(X_md.tolist(), columns=descriptor_names)
X_md = X_md.dropna()

y = tox["ld_50"]

3) Find common scaffolds and encode them as binary vecotrs (one-hot-encoding). Note that the Scaffolds are represented as SMILES and can be used for fingerprints later

In [5]:
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)

tox["scaffold"] = tox["smiles"].apply(get_scaffold).astype(str)

In [11]:
tox.head()

,smiles,ld_50,scaffold
0,[O-][N+](=Nc1ccccc1)c1ccccc1,2.505,c1ccc(N=[NH+]c2ccccc2)cc1
1,BrC(Br)Br,2.343,
2,C=CBr,2.330,
3,Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br,1.465,c1ccc(-c2ccccc2)cc1
4,S=C=Nc1ccc(Br)cc1,2.729,c1ccccc1


OneHotEncoder for the Murcko scaffolds:

Murcko scaffolds are molecules with ring systems and linkers, but no side chains. They are often used in drug discovery to identify common structural features among active compounds.

In [14]:
# Step 1: create the encoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Step 2: fit and transform the scaffold column
# tox[["scaffold"]] looks like:
# scaffold
# c1ccccc1
# c1ccccc1
# c1ncnc2ncnc12
# ...
X_scaf = ohe.fit_transform(tox[["scaffold"]])

# Step 3: make it a readable DataFrame
X_scaf = pd.DataFrame(X_scaf, columns=ohe.get_feature_names_out())

In [15]:
X_scaf.head()

,scaffold_,scaffold_C(#Cc1ccccc1)Cc1ccccc1,scaffold_C(=C(c1ccccc1)c1ccccc1)c1ccccc1,scaffold_C(=C1SCC(c2ccccc2)S1)n1ccnc1,scaffold_C(=CSSCC1CCCO1)NCc1cncnc1,scaffold_C(=Cc1ccccc1)C(C=Cc1ccccc1)=NN=C1NCCCN1,scaffold_C(=Cc1ccccc1)CN1CCN(c2cccnn2)CC1,scaffold_C(=Cc1ccccc1)c1ccccc1,scaffold_C(=Cc1ccco1)c1ccccn1,scaffold_C(=Cc1ccco1)c1ccco1,...,scaffold_c1ncn(C2CCCO2)n1,scaffold_c1ncnc(N(CCCCCCN(c2ncncn2)C2CCNCC2)C2CCNCC2)n1,scaffold_c1ncnc(NC2CC2)n1,scaffold_c1ncncn1,scaffold_c1ncsn1,scaffold_c1nnc(NCNc2nncs2)s1,scaffold_c1nnc[nH]1,scaffold_c1nnco1,scaffold_c1nncs1,scaffold_n1o[nH+]c2c1c1[nH+]onc1c1[nH+]onc21
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Instead of structural scaffolds (via SMARTS), encode functional groups using the one-hot-encoder. Add some more functional groups that you think have impact on the toxicity of compounds.

In [18]:
functional_groups = {
    "amine": "[NX3;H2,H1;!$(NC=O)]",
    "carboxylic_acid": "C(=O)[OX2H1]",
    "aromatic_ring": "a",
    "thiol":           "[SX2H]",
    "sulfonamide": "S(=O)(=O)N",
    "nitro":           "[N+](=O)[O-]",
    "halogen":         "[F,Cl,Br,I]",
    "epoxide":         "C1OC1",
    "aldehyde":        "[CX3H1](=O)",
    "thiol":           "[SX2H]"


}

In [19]:
patterns = {k: Chem.MolFromSmarts(v) for k,v in functional_groups.items()}

def detect_groups(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return {
        name: int(mol.HasSubstructMatch(pat))
        for name, pat in patterns.items()
    }

X_fg = pd.DataFrame(
    tox["smiles"].apply(detect_groups).tolist()
)

In [21]:
X_fg.head()

,amine,carboxylic_acid,aromatic_ring,thiol,sulfonamide,nitro,halogen,epoxide,aldehyde
0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,1,0,0
3,0,0,1,0,0,0,1,0,0
4,0,0,1,0,0,0,1,0,0


In [23]:

def evaluate(X, y, name):
    # align indices
    idx = X.index.intersection(y.index)
    X_, y_ = X.loc[idx], y.loc[idx]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_, y_, test_size=0.2, random_state=42
    )
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    r2 = r2_score(y_test, rf.predict(X_test))
    print(f"{name}: R² = {r2:.3f}")
    return r2

# build combinations
X_md_scaf     = pd.concat([X_md.reset_index(drop=True),
                            X_scaf.reset_index(drop=True)], axis=1)
X_md_scaf_fg  = pd.concat([X_md.reset_index(drop=True),
                            X_scaf.reset_index(drop=True),
                            X_fg.reset_index(drop=True)], axis=1)

# evaluate all 5
results = {}
results["X_md"]         = evaluate(X_md, y, "X_md")
results["X_scaf"]       = evaluate(X_scaf, y, "X_scaf")
results["X_fg"]         = evaluate(X_fg, y, "X_fg")
results["X_md_scaf"]    = evaluate(X_md_scaf, y, "X_md_scaf")
results["X_md_scaf_fg"] = evaluate(X_md_scaf_fg, y, "X_md_scaf_fg")

# pick the best
best_name = max(results, key=results.get)
print(f"\nBest: {best_name} with R² = {results[best_name]:.3f}")


X_md: R² = 0.619
X_scaf: R² = 0.207
X_fg: R² = 0.065
X_md_scaf: R² = 0.614
X_md_scaf_fg: R² = 0.614

Best: X_md with R² = 0.619


5) Pick the best one and prune the features regarding variance and correlation. Run the regression model again and compare the performance.

In [29]:
from sklearn.feature_selection import VarianceThreshold

# start with the winner
X_best = X_md.copy().fillna(0)
print(f"Before pruning: {X_best.shape[1]} features")

# Step 1 — remove low variance features
# since they don't vary much across samples, they don't add much information for the model
sel = VarianceThreshold(threshold=0.01)
X_pruned = pd.DataFrame(
    sel.fit_transform(X_best),
    columns=X_best.columns[sel.get_support()]
)
print(f"After variance filter: {X_pruned.shape[1]} features")

# Step 2 — remove highly correlated features 
# we remove them since they don't add new information and can cause overfitting
corr_matrix = X_pruned.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
X_pruned = X_pruned.drop(columns=to_drop)
print(f"After correlation filter: {X_pruned.shape[1]} features")

# Step 3 — run model again on cleaned X_md
evaluate(
    X_pruned.reset_index(drop=True),
    y.reset_index(drop=True),
    "X_md_pruned"
)
#calculate difference of filtered vs unfiltered
r2_unfiltered = results["X_md"]
r2_pruned = evaluate(
    X_pruned.reset_index(drop=True),
    y.reset_index(drop=True),
    "X_md_pruned"
)
print(f"Difference in R²: {r2_pruned - r2_unfiltered:.3f}")
#difference of features
print(f"Difference in features: {X_best.shape[1]- X_pruned.shape[1]}")

Before pruning: 217 features
After variance filter: 192 features
After correlation filter: 150 features
X_md_pruned: R² = 0.620
X_md_pruned: R² = 0.620
Difference in R²: 0.001
Difference in features: 67


6) Generate Morgan fingerprints (2048 bit, radius=2) for both "smiles" (entire molecules) and "scaffolds" (SMILES of common scaffolds only)

In [30]:
from rdkit.Chem import rdFingerprintGenerator

# create the Morgan fingerprint generator
# radius=2, 2048 bits — modern API (not the deprecated version)
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# --- Fingerprints for full molecules ---
fps_mol = []
valid_idx = []

for i, smi in enumerate(tox["smiles"]):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fps_mol.append(mfpgen.GetFingerprint(mol))
        valid_idx.append(i)

print(f"Valid molecules: {len(fps_mol)}")

# --- Fingerprints for scaffolds ---
fps_scaf = []
valid_idx_scaf = []

for i, smi in enumerate(tox["scaffold"].iloc[valid_idx]):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fps_scaf.append(mfpgen.GetFingerprint(mol))
        valid_idx_scaf.append(i)

print(f"Valid scaffolds: {len(fps_scaf)}")

Valid molecules: 7376
Valid scaffolds: 7376


7) Butina clustering (as in last exercise): Experiment with different cutoffs and filter limits for the clusters (compare in the visualisation, no need to rerun the UMAP visualisation in between)

In [ ]:
dists = []
nfps = len(fps)
cutoff = 0.6

for i in range(1, nfps):
    sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    dists.extend([1-x for x in sims])

clusters = Butina.ClusterData(
    dists,
    nfps,
    cutoff,
    isDistData=True
)

print("Number of clusters:", len(clusters))

In [ ]:
# filter out small clusters, rare chemoptypes, ...
clusters_filtered = [c for c in clusters if len(c) >= 5]

butina_labels = np.full(nfps, -1)
for cluster_id, cluster in enumerate(clusters_filtered):
    for id in cluster:
        butina_labels[id] = cluster_id

sizes = [len(c) for c in clusters_filtered]

print("clusters:", len(sizes))
print("mean size:", np.mean(sizes))
print("max size:", np.max(sizes))
print("singletons:", sum(s == 1 for s in sizes))

Calculate UMAP space - play with `n_neighbors` and `min_distance` until you are satisfied with the visualisation.

In [ ]:
# convert fingerprints to numpy
fp_array = np.zeros((nfps, 2048), dtype=int)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, fp_array[i])

In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

umap_emb = reducer.fit_transform(fp_array)

Visualise the UMAP space with the a) the cluster labels and b) the toxicity as colour hue. (Incomplete snippet for matplotlib provided - feel free to use whatever visualisation that works best for you)

In [ ]:
# ax1 = Chemical space and butina clustering
ax1.scatter(
    umap_emb[:,0],
    umap_emb[:,1],
    c=butina_labels,
    cmap="viridis",
    s=10,
    alpha=0.8
)
ax1.set(xlabel ="UMAP 1", ylabel="UMAP 2")
ax1.set_title("Butina clusters")


#### 9) Discussion points
1) How did the scaffolds and functional groups perform in the model performance in comparison the molecular descriptors? Comment on possible reasons. Which combination was suited best?
2) Is the One-Hot Encoding needed in this case?
3) Did the feature filtering (variance, correlation) have an improving effect? Explain!
- Feature pruning reduced the dimensionality by 31 % 67 features removed → model got simpler,
- R^2 remained nearly unchanged (0.619 → 0.620) → those features added nothing. This proves the removed features were redundant or uninformative
- A simpler model with fewer features is always preferred and easier to interpret, faster to train, less risk of overfitting on new molecules.

4) Consider to make this kind of workflow for feature comparison more generally usable: How would a clean solution look? How could you ensure compatibility with other models? How would the function / pipeline look schematically?
5) Visualisation: Using the scaffold fingerprints and the fingerprints for the entire molecule makes a huge difference. What is reason for this effect? Discuss this approach in contrast to similarity thresholds in the Butina clustering.
6) When might scaffolding prove useful?
7) Is toxicity correlated to molecular structure? I.e. what did the toxicity map in UMAP space reveal?
